# Multi-Region DFX Test Notebook

Tests the **two-region DFX** system — two independent reconfigurable regions (VS_0, VS_1) each holding one of two possible RMs.

| Region | Slot ID | Partial bitstreams |
|--------|---------|--------------------|
| hier_0 | 0       | `rm_0_region_0.bin`, `rm_1_region_0.bin` |
| hier_1 | 1       | `rm_0_region_1.bin`, `rm_1_region_1.bin` |

Results are read back from `axi_gpio_0` (region 0) and `axi_gpio_1` (region 1).

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import sys
import os

## Step 1 — Project Configuration

In [ ]:
from pathlib import Path

PRJ_DIR    = Path().resolve()          # notebook's cwd (reliable on PYNQ Jupyter)
PRJ_HW_DIR = PRJ_DIR / 'hw'
PRJ_SW_DIR = PRJ_DIR / 'sw'

FULL_BS_NAME   = 'system.bin'
DFX_CFG_FILE   = 'dfx_ctrl_cfg.txt'

AMT_REGION = 2   # VS_0 (slot 0) and VS_1 (slot 1)
AMT_RM     = 2   # RM_0 and RM_1 per region

# partial bitstream names[region_idx][rm_idx]
PAR_BS = [
    ['rm_0_region_0.bin', 'rm_1_region_0.bin'],   # slot 0 (region 0)
    ['rm_0_region_1.bin', 'rm_1_region_1.bin'],   # slot 1 (region 1)
]

print("HW dir:", PRJ_HW_DIR)
print("SW dir:", PRJ_SW_DIR)

## Step 2 — Import DFX Driver and Load Full Bitstream

Import the DFX driver **before** loading the overlay so PYNQ binds it automatically.

In [ ]:
sys.path.insert(0, PRJ_SW_DIR)
from sw.dfx_unified import DFX_Unified_Driver
import sw.cap as cap

In [ ]:
# Switch PL interface to PCAP for full bitstream load
cap.change_pl_config_mode("pcap", True, "")

overlay = Overlay(os.path.join(PRJ_HW_DIR, FULL_BS_NAME))
print("Overlay loaded")
print("IPs:", list(overlay.ip_dict.keys()))

## Step 3 — Get DFX Controller

PYNQ binds `system_dfx_controller_0_0` to `DFX_Unified_Driver` via `bindto`. The `dfx_ctrl` attribute wraps the register interface with multi-slot address support.

In [ ]:
dfx_unified = overlay.dfx_controller_0
dfx_ctrl    = dfx_unified.dfx_ctrl

## Step 4 — Configure DFX Controller

Parse the controller config file to retrieve the VS/bank/register address structure.
Then switch PL interface to **ICAP** so partial bitstreams can be loaded from the fabric.

In [ ]:
dfx_ctrl.config(os.path.join(PRJ_HW_DIR, DFX_CFG_FILE))
print(f"BLS_VSID   = {dfx_ctrl.BLS_VSID}")
print(f"BLS_BANKID = {dfx_ctrl.BLS_BANKID}")
print(f"BLS_REGID  = {dfx_ctrl.BLS_REGID}")

In [ ]:
# Switch PL interface to ICAP for partial reconfiguration
cap.change_pl_config_mode("icap", True, "")

## Step 5 — Reset All Slots

Shut down both VS slots to bring them to a clean idle state before programming metadata.

In [ ]:
for slot in range(AMT_REGION):
    dfx_ctrl.shutdown_engine(slot)

for slot in range(AMT_REGION):
    print(f"--- slot {slot} status ---")
    dfx_ctrl.print_status(slot)

## Step 6 — Allocate Partial Bitstreams in CMA

Load all partial bitstreams into physically contiguous memory.

```
cma_buf[slot][rm_idx]  →  (buffer, physical_address, size)
```

In [ ]:
cma_buf = []
for slot in range(AMT_REGION):
    slot_bufs = []
    for rm_idx in range(AMT_RM):
        bs_path = os.path.join(PRJ_HW_DIR, PAR_BS[slot][rm_idx])
        buf, phy_addr, size = dfx_ctrl.allocate_bit_stream_cma(bs_path)
        slot_bufs.append((buf, phy_addr, size))
        print(f"slot {slot} rm {rm_idx}: phy_addr={hex(phy_addr)}  size={size}")
    cma_buf.append(slot_bufs)

## Step 7 — Set DFX Controller Metadata

Register each partial bitstream address and size with the controller.
`set_simple_meta_data(slot_id, rm_idx, phy_addr, size)` programs:
- Trigger→RM mapping (RMM bank)
- RM control word (RMINFO bank)
- Bitstream address + size (BSINFO bank)

In [ ]:
for slot in range(AMT_REGION):
    for rm_idx in range(AMT_RM):
        _, phy_addr, size = cma_buf[slot][rm_idx]
        dfx_ctrl.set_simple_meta_data(slot, rm_idx, phy_addr, size)

## Step 8 — Verify Metadata

In [ ]:
for slot in range(AMT_REGION):
    print(f"\n=== slot {slot} ===")
    dfx_ctrl.print_status(slot)
    for rm_idx in range(AMT_RM):
        dfx_ctrl.print_simple_meta_data(slot, rm_idx)

## Step 9 — Load Initial RM into Each Region

Trigger RM_0 (trigger_id=0) into each slot so both regions have a valid bitstream before the experiment.

In [ ]:
INITIAL_RM = 0

for slot in range(AMT_REGION):
    dfx_ctrl.trig(slot, INITIAL_RM)
    dfx_ctrl.restart_no_status(slot)
    print(f"slot {slot}: triggered RM {INITIAL_RM}")

print("\n--- post-trigger status ---")
for slot in range(AMT_REGION):
    dfx_ctrl.print_status(slot)

## Step 10 — Read Results from GPIO

Read the output registers exposed by each reconfigurable region via AXI GPIO:
- `axi_gpio_0` → region 0 (slot 0)
- `axi_gpio_1` → region 1 (slot 1)

In [ ]:
gpio_0 = overlay.axi_gpio_0
gpio_1 = overlay.axi_gpio_1

val_region_0 = gpio_0.read()
val_region_1 = gpio_1.read()

print(f"region 0 (slot 0) output: {hex(val_region_0)}  ({val_region_0})")
print(f"region 1 (slot 1) output: {hex(val_region_1)}  ({val_region_1})")

## Step 11 — Swap RM and Re-read

Trigger RM_1 (trigger_id=1) into both regions and compare GPIO outputs.

In [ ]:
SWAP_RM = 1

for slot in range(AMT_REGION):
    dfx_ctrl.trig(slot, SWAP_RM)
    dfx_ctrl.restart_no_status(slot)
    print(f"slot {slot}: triggered RM {SWAP_RM}")

In [ ]:
val_region_0 = gpio_0.read()
val_region_1 = gpio_1.read()

print(f"region 0 (slot 0) output after RM {SWAP_RM}: {hex(val_region_0)}  ({val_region_0})")
print(f"region 1 (slot 1) output after RM {SWAP_RM}: {hex(val_region_1)}  ({val_region_1})")

## Step 12 — Swap RM and Re-read (REGION 0 - RM_1) (REGION 1 - RM_0)



In [1]:
CUR_REGION  = 0
CUR_SWAP_RM = 1

dfx_ctrl.trig(CUR_REGION, CUR_SWAP_RM)
dfx_ctrl.restart_no_status(CUR_REGION)
print(f"slot {CUR_REGION}: triggered RM {CUR_SWAP_RM}")

CUR_REGION  = 1
CUR_SWAP_RM = 0


dfx_ctrl.trig(CUR_REGION, CUR_SWAP_RM)
dfx_ctrl.restart_no_status(CUR_REGION)
print(f"slot {CUR_REGION}: triggered RM {SWAP_RM}")

NameError: name 'dfx_ctrl' is not defined

In [ ]:
val_region_0 = gpio_0.read()
val_region_1 = gpio_1.read()

print(f"region 0 (slot 0) output after RM {1}: {hex(val_region_0)}  ({val_region_0})")
print(f"region 1 (slot 1) output after RM {0}: {hex(val_region_1)}  ({val_region_1})")